# BM25 lexical search

Semantic search matches meaning, so it can miss an exact term: an incident ID, an error code,
a product name. **Lexical search** matches the words themselves. BM25 (Best Match 25) is the
classic algorithm:

1. **tokenize** the query: `"What happened with INC-2023-Q4-011?"` → `what`, `happened`, `with`, `inc`, `2023`, `q4`, `011`
2. **count** how often each term appears in each document
3. **weight** the terms: a rare term (`011`) counts a lot, a common one (`with`) almost nothing (IDF)
4. **rank** the documents by the weighted counts; repeats help less and less (`k1`), long documents are penalized (`b`)

No API calls, no embeddings. The class is in [`bm25_index.py`](./bm25_index.py). Combining BM25 with
semantic search is the next lesson: **hybrid search**.

## Setup

In [1]:
from pathlib import Path

from bm25_index import BM25Index
from chunking import chunk_by_section

chunks = chunk_by_section(Path("report.md").read_text())

store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

print(store)

BM25Index(count=15, k1=1.5, b=0.75)


## Tokens and their weights

The tokenizer lowercases and splits on anything that is not a letter or digit. IDF is the weight of a term.

In [2]:
query = "What happened with INC-2023-Q4-011?"
tokens = store._tokenizer(query)

for token in tokens:
    in_docs = sum(token in store._tokenizer(chunk) for chunk in chunks)
    print(f"{token:9} in {in_docs:2} of {len(chunks)} chunks, idf {store.idf(token):.2f}")

what      in  0 of 15 chunks, idf 3.47
happened  in  0 of 15 chunks, idf 3.47
with      in 10 of 15 chunks, idf 0.42
inc       in  2 of 15 chunks, idf 1.86
2023      in  2 of 15 chunks, idf 1.86
q4        in  2 of 15 chunks, idf 1.86
011       in  2 of 15 chunks, idf 1.86


`what` and `happened` are not in the report at all: the highest weight, but nothing to match.
`with` is in 10 of 15 chunks and weighs almost nothing. The ID splits into four tokens, each in 2 chunks.

## Search

The lesson's query. `search` returns `(document, distance)` like `VectorIndex`: the BM25 score is turned
into `exp(-0.1 * score)`, so smaller is better and the two indexes can be combined later. Chunks
without a single query term are left out.

In [3]:
results = store.search(query, k=3)

for doc, distance in results:
    print(f"{distance:.3f}\n{doc['content'][:200]}\n")

0.271
## Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core syst

0.332
## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `IN

0.939
## Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project upda



Both sections that mention the incident come first. The software section wins over the incident report
itself: it is shorter and names the ID twice.

## Lexical vs semantic

The rank of the right section for four questions (1 = first, `-` = not found). Two questions use exact IDs,
two use plain words. The vector index is the one from the previous lesson.

In [4]:
from embeddings import generate_embedding
from vector_index import VectorIndex

vector_store = VectorIndex(embedding_fn=generate_embedding)
vector_store.add_documents([{"content": chunk} for chunk in chunks])

# each question with the section that answers it
questions = {
    "What happened with INC-2023-Q4-011?": "## Section 10:",
    "TIMEOUT_QUERY_DB_0xDEADBEEF": "## Section 2:",
    "What did the software engineering dept do last year?": "## Section 2:",
    "Which team stopped hackers from breaking in?": "## Section 10:",
}


def rank_of(results, prefix):
    ranks = [rank for rank, (doc, _) in enumerate(results, start=1) if doc["content"].startswith(prefix)]
    return str(ranks[0]) if ranks else "-"


print(f"{'question':54} {'vector':>6} {'bm25':>5}")
for question, answer in questions.items():
    vector_rank = rank_of(vector_store.search(question, k=len(chunks)), answer)
    bm25_rank = rank_of(store.search(question, k=len(chunks)), answer)
    print(f"{question:54} {vector_rank:>6} {bm25_rank:>5}")

question                                               vector  bm25


What happened with INC-2023-Q4-011?                         1     2


TIMEOUT_QUERY_DB_0xDEADBEEF                                 1     1


What did the software engineering dept do last year?        2     3


Which team stopped hackers from breaking in?                1     -


- **The IDs**: BM25 finds them, as expected. The course shows semantic search missing the incident ID;
  here `voyage-4-large` finds it too, so the gap depends on the embedding model.
- **"stopped hackers from breaking in"**: no word in common with the incident report ("intrusion attempt",
  "contained"), so BM25 cannot find it; semantic search does.
- **Common words add noise**: BM25 still returns chunks for the hacker question, matched on `which`,
  `from`, `in`. A tokenizer can drop such stop words:

In [5]:
import re

STOP_WORDS = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "from", "with", "by",
              "is", "was", "what", "which", "who", "did", "do", "last", "this", "year"}


def tokenize_without_stop_words(text):
    return [token for token in re.split(r"\W+", text.lower()) if token and token not in STOP_WORDS]


filtered_store = BM25Index(tokenizer=tokenize_without_stop_words)
filtered_store.add_documents([{"content": chunk} for chunk in chunks])

for question, answer in questions.items():
    before = rank_of(store.search(question, k=len(chunks)), answer)
    after = rank_of(filtered_store.search(question, k=len(chunks)), answer)
    print(f"{question:54} {before:>3} -> {after:>3}   tokens: {tokenize_without_stop_words(question)}")

What happened with INC-2023-Q4-011?                      2 ->   2   tokens: ['happened', 'inc', '2023', 'q4', '011']
TIMEOUT_QUERY_DB_0xDEADBEEF                              1 ->   1   tokens: ['timeout_query_db_0xdeadbeef']
What did the software engineering dept do last year?     3 ->   2   tokens: ['software', 'engineering', 'dept']
Which team stopped hackers from breaking in?             - ->   -   tokens: ['team', 'stopped', 'hackers', 'breaking']


The software question moves up: without `the`, `did`, `last`, `year`, the Executive Summary loses
the points it got for repeating them. The hacker question is still out of reach: the words are just not there.

Next lesson: run both indexes and merge the results.